# GRASPED: Graph Anomaly Detection using Autoencoder with Spectral Encoder and Decoder

## Full Recreation from the Paper (arXiv:2508.15633)

This notebook recreates the GRASPED model exactly as described in the paper:
- **Encoder**: Wavelet-based Graph Neural Network (GWNN) using Haar multi-resolution analysis (Section 4.1)
- **Decoder 1 (Structure)**: MLP that predicts node degrees (Section 4.2.1)
- **Decoder 2 (Neighbor)**: MLP that predicts neighbor feature distributions via KL divergence (Section 4.2.2)
- **Decoder 3 (Attribute)**: Wiener Graph Deconvolutional Network (GDN) that reconstructs node features (Section 4.2.3)

The anomaly score for each node is its total weighted reconstruction loss (Equation 20).

### Architecture overview (Figure 1 in paper):

```
Input Graph G(A, X)
        |
        v
  [Wavelet-based Graph Encoder]  ---- 2 layers, hidden_dim=32
        |                               Each layer: H^(i) = sigma(M * H^(i-1) * W^(i-1))  (Eq. 8)
        |                               M = U * G_c * U^T  (multiscale diffusion operator, Eq. 7)
        v
  Latent Embedding h_u^(Z)
        |
  +-----+-----+------+
  |           |            |
  v           v            v
[Structure  [Neighbor   [Attribute
 Decoder]   Decoder]    Decoder (GDN)]
  |           |            |
  v           v            v
 L_d (Eq10) L_n (Eq14)  L_x (Eq19)
  |           |            |
  +-----+-----+------+
        |
        v
  Total Loss L = lambda_d*L_d + lambda_n*L_n + lambda_x*L_x  (Eq. 20)
```

In [1040]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import scipy.sparse as sp
from sklearn.metrics import roc_auc_score
import warnings
import time
import torch
from torch_geometric.data.storage import GlobalStorage

# Explicitly allow PyG objects to be unpickled
torch.serialization.add_safe_globals([GlobalStorage])

warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

Using device: cpu
PyTorch version: 2.10.0+cpu


---
## 1. Dataset Loading

The paper uses 5 real-world datasets with **organic** anomalies (Table 1):

| Dataset | Nodes | Edges | Features | Avg Degree | Anomalies | Ratio |
|---------|-------|-------|----------|------------|-----------|-------|
| Weibo   | 8,405 | 407,963 | 400 | 48.5 | 868 | 10.3% |
| Reddit  | 10,984| 168,016 | 64  | 15.3 | 366 | 3.3%  |
| Disney  | 124   | 335     | 28  | 2.7  | 6   | 4.8%  |
| Books   | 1,418 | 3,695   | 21  | 2.6  | 28  | 2.0%  |
| Enron   | 13,533| 176,987 | 18  | 13.1 | 5   | 0.04% |

These come from the PyGOD / BOND benchmark. We provide two loading options:
1. **`load_dataset_pygod`** - downloads from PyGOD (needs internet to `raw.githubusercontent.com`)
2. **`generate_synthetic_dataset`** - creates a synthetic graph matching the paper's statistics

**For exact reproduction of paper results, use option 1 with the real datasets.** Option 2 validates the code works.

In [1041]:
# Option 1: Load real datasets from PyGOD
# Uncomment and run if you have internet access to raw.githubusercontent.com
#
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pygod', '-q'])

def load_dataset_pygod(name):
    from pygod.utils import load_data
    data = load_data(name)
    edge_index = data.edge_index.numpy()
    features = data.x.numpy().astype(np.float32)
    labels = (data.y.numpy() > 0).astype(np.int64)
    n = features.shape[0]
    row, col = edge_index[0], edge_index[1]
    adj = sp.csr_matrix((np.ones(len(row)), (row, col)), shape=(n, n))
    adj = adj + adj.T; adj[adj > 1] = 1; adj.setdiag(0); adj.eliminate_zeros()
    return adj, features, labels

In [ ]:
# load dataset 
def load_dataset(name):
    """Try PyGOD first; fall back to synthetic."""
    STATS = {
        'weibo':  (8405, 407963, 400, 0.103),
        'reddit': (10984, 168016, 64, 0.033),
        'disney': (124, 335, 28, 0.048),
        'books':  (1418, 3695, 21, 0.020),
        'enron':  (13533, 176987, 18, 0.0004),
    }
    try:
        from pygod.utils import load_data
        data = load_data(name)
        edge_index = data.edge_index.numpy()
        features = data.x.numpy().astype(np.float32)
        labels = (data.y.numpy() > 0).astype(np.int64)
        n = features.shape[0]
        row, col = edge_index[0], edge_index[1]
        adj = sp.csr_matrix((np.ones(len(row)), (row, col)), shape=(n, n))
        adj = adj + adj.T; adj[adj > 1] = 1; adj.setdiag(0); adj.eliminate_zeros()
        source = "PyGOD (real)"
    except Exception:
        print(f"Failed to load {name} from PyGOD, generating synthetic data instead.")
    print(f"Dataset: {name} ({source})")
    print(f"  Nodes: {features.shape[0]}, Edges: {adj.nnz // 2}, Features: {features.shape[1]}")
    print(f"  Anomalies: {labels.sum()} ({100 * labels.mean():.2f}%)")
    return adj, features, labels

# Quick test
#adj_test, feat_test, lab_test = load_dataset('disney')

---
## 2. Graph Spectral Computations (Section 3)

The **normalized Laplacian**: $L = I - D^{-1/2} A D^{-1/2}$

Its eigendecomposition $L = U \Lambda U^\top$ gives:
- $U$ = eigenvector matrix (graph Fourier basis)
- $\Lambda = \text{diag}(\lambda_1, \dots, \lambda_n)$ with $\lambda_i \in [0, 2]$

**Spectral filtering**: $g_c(L)X = U \cdot g_c(\Lambda) \cdot U^\top X$

In [ ]:
def compute_laplacian_and_eigen(adj):
    """
    Compute normalized Laplacian L = I - D^{-1/2} A D^{-1/2}
    and its full eigendecomposition L = U * Lambda * U^T.
    Eigenvalues are clipped to [0, 2] for numerical stability.
    """
    n = adj.shape[0]
    # my adj consist of weights not just 1s and 0s
    un_adj = (adj > 0).astype(np.float32) # wherever adj is nonzero, set to 1 else 0
    degrees = np.array(un_adj.sum(axis=1)).flatten()
    deg_inv_sqrt = np.where(degrees > 0, 1.0 / np.sqrt(degrees), 0.0)
    D_inv_sqrt = sp.diags(deg_inv_sqrt)
    A_norm = D_inv_sqrt @ un_adj @ D_inv_sqrt
    L_sparse = sp.eye(n) - A_norm # normalised Laplacian
    eigenvalues, eigenvectors = np.linalg.eigh(L_sparse.toarray())
    eigenvalues = np.clip(eigenvalues, 0, 2) # clip eigenvals bvetween 0 and 2 for stabiolity
    return L_sparse, eigenvalues, eigenvectors


def sparse_to_torch(L_sparse):
    """Convert scipy sparse matrix to PyTorch sparse tensor."""
    L_coo = L_sparse.tocoo()
    indices = torch.tensor(np.vstack([L_coo.row, L_coo.col]), dtype=torch.long)
    values = torch.tensor(L_coo.data, dtype=torch.float32)
    return torch.sparse_coo_tensor(indices, values, L_coo.shape)

print("Spectral computation functions defined.")

Spectral computation functions defined.


---
## 3. Haar Wavelet Multi-Resolution Analysis (Section 4.1)

The key innovation of the encoder. The eigenvalue range [0, 2] represents frequencies:
- lambda ~ 0: low frequency (smooth signals)
- lambda ~ 2: high frequency (rapidly-varying signals)

Anomalies cause a **right-shift** in spectral energy (Tang et al. [31]).
GCN (low-pass filter) would miss this!

The **Haar wavelet** divides [0, 2] into K equal sub-bands, each with a
**learnable weight** theta_k. This creates an adaptive **band-pass filter**.

From **Equation 5**: $g_c(\lambda) = \sum_{k=0}^{K-1} \theta_{J,k} \cdot \phi_{\text{Haar},J,k}(\lambda)$

Where phi_Haar(lambda) = sqrt(K/2) if lambda is in sub-band k, else 0.

In [1044]:
# def compute_haar_basis(eigenvalues, K):
#     """
#     Compute Haar scaling function values at each eigenvalue (Equation 5).

#     Divides [0, 2] into K equal sub-bands. For each eigenvalue its bin gets
#     value 1.0, all others 0.0. So gc(lambda_i) = theta_{bin(lambda_i)}.

#     FIX: Previously used sqrt(K/2) normalization which inflated filter
#     values by sqrt(K/2). For K=128 that is 8x per layer => 64x after
#     2 layers => NaN. Now use plain indicator (1.0) so theta learns scaling.
#     """
#     n = len(eigenvalues)
#     haar_matrix = np.zeros((n, K), dtype=np.float32)
#     interval_width = 2.0 / K
#     for i, lam in enumerate(eigenvalues):
#         k = min(int(lam / interval_width), K - 1)
#         haar_matrix[i, k] = 1.0   # plain indicator: 1 if in bin k, else 0
#     return haar_matrix

# print("Haar basis fixed: one-hot rows, no sqrt normalization")


In [ ]:
def compute_haar_basis(eigenvalues, K):
    n = len(eigenvalues)
    haar_matrix = np.zeros((n, K), dtype=np.float32)
    interval_width = 2.0 / K
    norm_factor = np.sqrt(K / 2.0)
    for i, lam in enumerate(eigenvalues):
        k = min(int(lam / interval_width), K - 1)
        haar_matrix[i, k] = 1
    return haar_matrix

# def compute_haar_basis(eigenvalues, K):
#     """
#     Computes a strict orthogonal MRA Haar basis: V_0 + W_0 + W_1 ... + W_J-1
#     """
#     N = len(eigenvalues)
#     max_val = 2.0
#     basis_columns = []
    
#     # 1. Coarsest Father Scaling Function (V_0) spanning the whole [0, 2]
#     phi_0 = np.full(N, np.sqrt(1.0 / max_val), dtype=np.float64)
#     basis_columns.append(phi_0)
#     J = int(np.log2(2 * K))  # K = 2^(J-1) => J = log2(2K)
#     # Mother Wavelet Functions (W_0 to W_{J-1}) for finer details until J-1
#     for j in range(1, J):
#         num_shifts = 2 ** (j-1)
#         interval_width = max_val / num_shifts
#         half_width = interval_width / 2.0
#         norm_factor = np.sqrt(num_shifts / max_val)

#         for k in range(num_shifts):
#             left = k * interval_width
#             mid = left + half_width
#             right = left + interval_width

#             psi = np.zeros(N, dtype=np.float64)
#             psi[(eigenvalues >= left) & (eigenvalues < mid)] = norm_factor
#             psi[(eigenvalues >= mid) & (eigenvalues < right)] = -norm_factor
#             basis_columns.append(psi)

#     basis = np.stack(basis_columns, axis=1) # size (N, D) where D = 1 + sum_{j=1}^{J-1} 2^(j-1) = 2^J - 1
#     D = basis.shape[1]
#     haar_matrix = np.zeros((N, D), dtype=np.float32) # size (N, D)
#     for i in range(N):
#         haar_matrix[i, :] = basis[i, :]
#     return haar_matrix

# def compute_haar_basis(eigenvalues, K):
#     """
#     Args:
#         eigenvalues: (N,) graph Laplacian eigenvalues in [0, 2).
#         J: number of wavelet decomposition levels.
#     Returns:
#         B: (N, D) Haar basis matrix, where D = 2^J + (2^J - 1).
#     """
#     J = np.log2(2*K).astype(int)  # K = 2^(J-1)
#     N = len(eigenvalues)
#     max_val = 2.0

#     basis_columns = []
#     # PART 1 — Wavelet (mother) basis functions: psi_{j,k}

#     for j in range(1, J):
#         num_shifts = 2 ** (j-1)
#         interval_width = max_val / num_shifts
#         half_width = interval_width / 2.0
#         norm_factor = np.sqrt(num_shifts / max_val)

#         for k in range(num_shifts):
#             left = k * interval_width
#             mid = left + half_width
#             right = left + interval_width

#             psi = np.zeros(N, dtype=np.float64)
#             psi[(eigenvalues >= left) & (eigenvalues < mid)] = norm_factor
#             psi[(eigenvalues >= mid) & (eigenvalues < right)] = -norm_factor
#             basis_columns.append(psi)

#     # PART 2 — Scaling (father) basis functions: phi_{J,k}

#     num_shifts_J = 2 ** J
#     interval_width = max_val / num_shifts_J
#     norm_factor = np.sqrt(num_shifts_J / max_val)

#     for k in range(num_shifts_J):
#         left = k * interval_width
#         right = left + interval_width

#         phi = np.zeros(N, dtype=np.float64)
#         phi[(eigenvalues >= left) & (eigenvalues < right)] = norm_factor
#         basis_columns.append(phi)

#     # Stack into the Haar basis matrix B: shape (N, D)

#     basis = np.stack(basis_columns, axis=1)
#     return torch.tensor(basis, dtype=eigenvalues.dtype, device=eigenvalues.device)

---
## 4. Wavelet Encoder (Section 4.1, Equation 8)

2-layer encoder. Each layer: $H^{(i)} = \text{ReLU}(M \cdot H^{(i-1)} \cdot W^{(i-1)})$

Where M = U * diag(g_c(lambda_1),...) * U^T is the Multiscale Diffusion Operator (Eq. 7),
and each layer has its own learnable theta coefficients.

In [1046]:
class WaveletEncoder(nn.Module):
    """
    2-layer Wavelet-based Graph Encoder
    Uses Haar MRA for learnable band-pass filtering.
    """
    def __init__(self, input_dim, hidden_dim, K, eigenvalues, eigenvectors):
        super().__init__()
        # Precomputed Haar basis (not learnable)
        self.register_buffer('haar_matrix', 
                             torch.tensor(compute_haar_basis(eigenvalues, K), dtype=torch.float32))
        # Eigenvector matrix U (.copy() needed for negative-stride numpy arrays)
        self.register_buffer('U', torch.tensor(eigenvectors.copy(), dtype=torch.float32))
        
        # Learnable filter coefficients theta for each layer (Eq. 5)
        # Init to ones: gc(lambda)=1 initially = identity-like filter, stable start
        # torch.randn(K) * 0.01
        #norm_factor = np.sqrt(K / 2.0)
        self.theta1 = nn.Parameter(torch.ones(K))  # Layer 1
        self.theta2 = nn.Parameter(torch.ones(K))  # Layer 2
        #self.theta1 = nn.Parameter(torch.randn(K)*0.01)  # Layer 1
        #self.theta2 = nn.Parameter(torch.randn(K) * 0.01)  # Layer 2
        #self.K = K
        # Learnable weight matrices W^(i) (Eq. 8)
        self.W1 = nn.Linear(input_dim, hidden_dim, bias=False)
        self.W2 = nn.Linear(hidden_dim, hidden_dim, bias=False)
    
    def _build_M(self, theta):
        """
        Build Multiscale Diffusion Operator M (Equations 5-7).
        1. g_c(lambda_i) = haar_matrix @ theta  (Eq. 5: filter response)
        2. M = U * diag(g_c) * U^T              (Eq. 7: vertex-domain operator)
        """
        # Clamp theta to [-3,3]: prevents gc(lambda) from exploding
        gc = self.haar_matrix @ theta
        #.clamp(-10.0, 10.0)   # filter values at each eigenvalue
        return (self.U * gc.unsqueeze(0)) @ self.U.T  # (n, n)
    
    def forward(self, X):
        """Eq. 8: H^(i) = ReLU(M_i @ H^(i-1) @ W_i). H^(0) = X."""
        # bound = np.sqrt(6.0 / self.K)  # Xavier uniform: sqrt(6 / (fan_in + fan_out)), fan_out=1
        # nn.init.uniform_(self.theta1, -bound, bound)
        # nn.init.uniform_(self.theta2, -bound, bound)
        H_1 = F.relu(self.W1(self._build_M(self.theta1) @ X))  # Layer 1
        H = F.relu(self.W2(self._build_M(self.theta2) @ H_1))  # Layer 2
        return H  # latent embedding h_u^(Z), shape (n, hidden_dim)

print("WaveletEncoder defined.")

WaveletEncoder defined.


---
## 5. Structure Decoder (Section 4.2.1)

$\hat{d}_u = \phi_{str}(h_u^{(Z)})$ (Eq. 9), Loss: $L_u^d = \|\hat{d}_u - d_u\|_2^2$ (Eq. 10)

Anomalous nodes often have unusual degree patterns (Table 2: Weibo anomalies -51% degree).

In [1047]:
class StructureDecoder(nn.Module):
    """MLP predicting node degrees from latent embeddings."""
    def __init__(self, hidden_dim):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, 1))
            #nn.Linear(hidden_dim, 16), nn.Linear(16, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, 1)
        )
    def forward(self, H):
        return self.mlp(H)  # (n, 1)

print("StructureDecoder defined.")

StructureDecoder defined.


---
## 6. Neighbor Decoder (Section 4.2.2)

Predicts the distribution of neighbors' features. Two MLPs output mean and log-variance:

$\hat{\mu}_u = \phi_\mu(h_u^{(Z)})$, $\hat{\Sigma}_u = \text{diag}(\exp(\phi_\sigma(h_u^{(Z)})))$ (Eq. 11)

Empirical stats computed from actual neighbors (Eq. 12-13). Loss = KL divergence (Eq. 14).

In [1048]:
class NeighborDecoder(nn.Module):
    """Predicts neighbor feature distribution (mean + diagonal covariance)."""
    def __init__(self, hidden_dim, feature_dim):
        super().__init__()
        self.phi_mu = nn.Sequential(
            nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, feature_dim))
            #nn.Linear(hidden_dim, 16), nn.Linear(16, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, feature_dim)
        )
        self.phi_sigma = nn.Sequential(
            nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, feature_dim))
            #nn.Linear(hidden_dim, 16), nn.Linear(16, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, feature_dim)
        )
    def forward(self, H):
        return self.phi_mu(H), self.phi_sigma(H)  # mu_hat, log_sigma_hat


def compute_neighbor_stats(adj, features, S): # S = max neighbours to use per node for neighbour stats
    """
    Compute empirical neighbor mean (Eq. 12) and log-variance (Eq. 13).
    S = sample size: max neighbors to use per node (Table 4 hyperparameter).
    """
    n, d = features.shape
    mu = np.zeros((n, d), dtype=np.float32)
    log_sigma = np.zeros((n, d), dtype=np.float32)
    eps = 1e-6
    adj_csr = adj.tocsr()
    
    for u in range(n):
        neighbors = adj_csr[u].indices
        if len(neighbors) == 0:
            mu[u] = features[u]; log_sigma[u] = np.log(eps); continue
        sampled = np.random.choice(neighbors, size=min(len(neighbors), S), replace=False)
        nf = features[sampled]
        mu[u] = nf.mean(axis=0)
        log_sigma[u] = np.log(nf.var(axis=0, ddof=1) + eps) if len(sampled) > 1 else np.log(eps)
        #log_sigma[u] = np.log(nf.var(axis=0) + eps) if len(sampled) > 1 else np.log(eps)
    return mu, log_sigma

print("NeighborDecoder and neighbor stats defined.")

NeighborDecoder and neighbor stats defined.


---
## 7. Wiener GDN Attribute Decoder (Section 4.2.3)

The most sophisticated decoder. Reverses the encoder's convolution using **Wiener deconvolution**.

**Wiener filter** (Eq. 16): $g_w(\lambda_i) = \frac{g_c(\lambda_i)}{g_c^2(\lambda_i) + \sigma^2 / E[x_i^{*2}]}$

Where $g_c(\lambda) = e^{-\lambda}$ (Heat kernel). The AER term ($\sigma^2/E[x_i^{*2}]$) balances:
- High AER -> suppress (noise dominates) 
- Low AER -> preserve (signal dominates)

**Remez polynomial** approximation (Eq. 17): $D_\gamma = \sum_k c_k L^k$ avoids full eigendecomposition.

**Noise augmentation**: $\hat{H} = H + \beta E$, $E \sim N(0, \sigma_P^2 I)$

**Multi-channel** (Eq. 18): Q channels per layer, aggregated by summation.

In [1049]:
def compute_wiener_cheby_coefficients(aer, K_poly):
    """
    Fit Chebyshev polynomial approximation of Wiener filter (paper Eq 17).

    Wiener filter: gw(lambda) = exp(-lambda) / (exp(-2*lambda) + AER)
    on lambda in [0, 2].

    We map [0,2] -> [-1,1] via t = lambda - 1, then fit in Chebyshev basis.
    Chebyshev basis is ALWAYS numerically stable regardless of degree K.
    This matches the paper's "Chebyshev node interpolation" description.

    Returns Chebyshev coefficients [c_0, c_1, ..., c_K] for T_k(t).
    """
    # Chebyshev nodes on [-1, 1]: t_j = cos((2j+1)*pi / (2K+2))
    j = np.arange(K_poly + 1)
    t_nodes = np.cos((2 * j + 1) * np.pi / (2 * K_poly + 2))  # in [-1, 1]
    # Map back to [0, 2]: lambda = t + 1
    lam_nodes = t_nodes + 1.0
    # Wiener filter values (Eq 16): gw = gc / (gc^2 + AER)
    gc = np.exp(-lam_nodes) # using g_c =e^-lambda_i heat kernel for conv as in paper
    gw = gc / ((gc ** 2 )+ aer + 1e-8)
    # Fit in Chebyshev basis (chebfit works in the T_k basis on [-1,1])
    coeffs = np.polynomial.chebyshev.chebfit(t_nodes, gw, K_poly) # gives me K_poly+1 coeffs for T_0, T_1, ..., T_K
    return coeffs.astype(np.float32)   # shape (K_poly+1,)


def apply_chebyshev_filter(L_sparse, x, cheby_coeffs):
    """
    Apply Chebyshev polynomial filter to x using the stable recurrence (Eq 17).

    For L with eigenvalues in [0,2], map to L' = L - I (eigenvalues in [-1,1]).
    Chebyshev recurrence: T_0(L')x = x, T_1(L')x = L'x,
                           T_k(L')x = 2 L' T_{k-1} - T_{k-2}

    This is provably O(K*|E|) and numerically stable for any K,
    because ||L'|| <= 1 so no power explosion.
    """
    K = len(cheby_coeffs) - 1
    # T_0(L') x = x
    T_prev = x
    # T_1(L') x = (L - I) x = L x - x
    Lx = torch.sparse.mm(L_sparse, x)
    T_curr = Lx - x
    # Accumulate: result = c_0 T_0 + c_1 T_1 + ...
    result = cheby_coeffs[0] * T_prev + cheby_coeffs[1] * T_curr
    for k in range(2, K + 1):
        # T_k(L') x = 2 L' T_{k-1} x - T_{k-2} x
        #             = 2 (L - I) T_{k-1} x - T_{k-2} x
        T_next = 2.0 * (torch.sparse.mm(L_sparse, T_curr) - T_curr) - T_prev 
        #T_next = 2.0 * (torch.sparse.mm(L_sparse, T_curr)) - T_prev 
        result = result + cheby_coeffs[k] * T_next
        T_prev, T_curr = T_curr, T_next
    return result


class WienerGDNDecoder(nn.Module):
    """
    Wiener Graph Deconvolutional Network for attribute reconstruction.
    Z layers (=2, matching encoder), Q channels per layer.

    for filter diversity, matching the Wiener GDN paper [4].

    Polynomial order K is taken from the GRASPED hyperparameter table (same K
    as encoder). Applied via Chebyshev recurrence for numerical stability.
    """
    def __init__(self, hidden_dim, output_dim, beta=0.5, K_poly=8, Q=1, n_layers=2):
        super().__init__()
        self.beta = beta
        self.Q = Q
        self.n_layers = n_layers
        self.K_poly = K_poly  # polynomial degree = K from hyperparams (no artificial cap)
        # Learnable weights per (layer, channel)
        self.weights = nn.ModuleList()
        for layer in range(n_layers):
            lw = nn.ModuleList()
            out_dim = hidden_dim if layer < n_layers - 1 else output_dim
            for q in range(Q):
                lw.append(nn.Linear(hidden_dim, out_dim, bias=False))
            self.weights.append(lw)

    def forward(self, H, L_sparse, U, eigenvalues):
        sigma_P = H.var(unbiased=False).sqrt().detach().clamp(min=1e-8)
        if self.training:
            cur = H + torch.randn_like(H) * self.beta * sigma_P
        else:
            cur = H
        
        noise_var = (self.beta * sigma_P) ** 2
        
        # Per-eigenvalue spectral energy (Eq. 16: x_i* = U^T @ signal)
        H_spectral = U.T @ H.detach()                    # (n, hidden_dim)
        spectral_energy = (H_spectral ** 2).mean(dim=1)  # (n,) — energy per eigenvalue
        spectral_energy = spectral_energy.clamp(min=1e-10)
        
        # Per-eigenvalue AER (Eq. 16)
        aer_per_eig = (noise_var / spectral_energy).cpu().numpy()  # (n,)
        
        # Per-eigenvalue Wiener filter values
        eigs_np = eigenvalues  # numpy array
        gc_vals = np.exp(-eigs_np)
        gw_vals = gc_vals / (gc_vals**2 + aer_per_eig + 1e-8)  # (n,)
        
        # Fit polynomial to these N specific (lambda_i, gw_i) points
        t_vals = eigs_np - 1.0  # map [0,2] → [-1,1]
        coeffs_np = np.polynomial.chebyshev.chebfit(t_vals, gw_vals, self.K_poly)
        coeffs = torch.tensor(coeffs_np.astype(np.float32), device=H.device)
        
        # Apply the polynomial filter (same Chebyshev recurrence)
        for layer in range(self.n_layers):
            outs = []
            for q in range(self.Q):
                filtered = apply_chebyshev_filter(L_sparse, cur, coeffs)
                t = self.weights[layer][q](filtered)
                outs.append(F.relu(t) if layer < self.n_layers - 1 else t)
            cur = sum(outs)
        return cur

    # def forward(self, H, L_sparse):
    #     """Reconstruct attributes via multi-layer Wiener deconvolution (Eq. 18)."""
    #     # Noise injection: H_hat = H + beta * E,  E ~ N(0, sigma_P^2 * I)  (Eq 18 preamble)
    #     # sigma_P^2 = VAR[H^(Z)] as stated in the paper
    #     sigma_P = H.var(unbiased=False).sqrt().detach().clamp(min=1e-8)
    #     if self.training:
    #         cur = H + torch.randn_like(H) * self.beta * sigma_P
    #     else:
    #         cur = H
    #     # AER = sigma^2 / E[x*^2]  (Eq. 16) — computed directly, no clipping
    #     noise_var = float((self.beta * sigma_P) ** 2) + 1e-10
    #     signal_energy = float((H.detach() ** 2).mean()) + 1e-10
    #     aer_base = float(np.clip(noise_var / signal_energy, 1e-6, 1e6))   # AER (Eq. 16)

    #     # Multi-layer deconvolution (Eq. 18): Z layers, Q channels per layer
    #     for layer in range(self.n_layers):
    #         outs = []
    #         for q in range(self.Q):
    #             # Wiener kernel D_gamma approximated via polynomial (Eq. 17)
    #             coeffs_np = compute_wiener_cheby_coefficients(aer_base, self.K_poly)
    #             coeffs = torch.tensor(coeffs_np, device=H.device, dtype=torch.float32)
    #             filtered = apply_chebyshev_filter(L_sparse, cur, coeffs)
    #             t = self.weights[layer][q](filtered)
    #             # phi = ReLU for all but last layer; last layer is linear (reconstruction)
    #             outs.append(F.relu(t) if layer < self.n_layers - 1 else t)
    #         cur = sum(outs)  # AGG = summation (Eq. 18)
    #     return cur

print("WienerGDNDecoder: Chebyshev basis + AER (Eq. 16, Eq. 18)")



WienerGDNDecoder: Chebyshev basis + AER (Eq. 16, Eq. 18)


---
## 8. Complete GRASPED Model

In [1050]:
class GRASPED(nn.Module):
    """
    Graph Autoencoder with Spectral Encoder and Decoder.
    Anomaly score = weighted sum of 3 reconstruction losses (Eq. 20).
    """
    def __init__(self, input_dim, hidden_dim, K, beta, eigenvalues, eigenvectors):
        super().__init__()
        self.eigenvalues= torch.tensor(eigenvalues, dtype=torch.float32)
        self.eigenvectors = torch.tensor(eigenvectors, dtype=torch.float32)
        self.encoder = WaveletEncoder(input_dim, hidden_dim, K, eigenvalues, eigenvectors)
        self.str_dec = StructureDecoder(hidden_dim)
        self.nbr_dec = NeighborDecoder(hidden_dim, input_dim)
        # Pass K as polynomial degree to decoder (paper uses same K for encoder+decoder)
        self.attr_dec = WienerGDNDecoder(hidden_dim, input_dim, beta=beta, K_poly=K, Q=3)
    
    def forward(self, X, L_torch):
        H = self.encoder(X)
        mu_hat, log_sig_hat = self.nbr_dec(H)
        return {
            'deg': self.str_dec(H),        # (n, 1)
            'mu': mu_hat,                   # (n, d)
            'ls': log_sig_hat,              # (n, d)
            'xr': self.attr_dec(H, L_torch, self.eigenvectors, self.eigenvalues) # (n, d)
        }

print("GRASPED model defined.")

GRASPED model defined.


---
## 9. Loss Functions (Section 4.3, Equation 20)

1. **Degree loss** (Eq. 10): $L_u^d = \|\hat{d}_u - d_u\|^2$
2. **Neighbor KL loss** (Eq. 14): KL between predicted and empirical Gaussians
3. **Attribute loss** (Eq. 19): $L_u^x = \|x_u - \hat{x}_u\|_2$

Total: $L'_u = \lambda_d L_u^d + \lambda_n L_u^n + \lambda_x L_u^x$

In [1051]:
def compute_losses(out, X, deg_t, nmu_t, nls_t, ld, ln, lx):
    """
    Compute per-node losses and anomaly scores (Eq. 20).
    L_u = lambda_d * L^d_u + lambda_n * L^n_u + lambda_x * L^x_u
    """
    # Loss 1: Degree reconstruction (Eq. 10)
    L_d = (out['deg'].squeeze() - deg_t) ** 2

    # Loss 2: Neighbor KL divergence (Eq. 14, diagonal Gaussian)
    ls_hat = out['ls'].clamp(-50, 50)     # clamp log-variance for stability THIS IS WHAT WAS CAUSING NANs
    #ls_hat = out['ls']     # no clamping, let the model learn appropriate scale
    sh = torch.exp(ls_hat).clamp(min=1e-4)     # predicted variance (Sigma_hat diagonal)
    st = torch.exp(nls_t).clamp(min=1e-4)      # empirical variance (Sigma diagonal)
    #kl = 0.5 * (ls_hat - nls_t - 1 + st / sh + (nmu_t - out['mu']).clamp(-50, 50) ** 2 / sh).sum(dim=1)
    kl = 0.5 * (ls_hat - nls_t - 1 + st / sh + (nmu_t - out['mu']) ** 2 / sh).sum(dim=1)
    L_n = kl.clamp(min=0)  # KL >= 0 always

    # Loss 3: Attribute reconstruction (Eq. 19)
    L_x = torch.norm(X - out['xr'], dim=1)

    # normalise the 3 losses
    L_d = (L_d - L_d.min()) / (L_d.max() - L_d.min() + 1e-10)
    L_n = (L_n - L_n.min()) / (L_n.max() - L_n.min() + 1e-10)
    L_x = (L_x - L_x.min()) / (L_x.max() - L_x.min() + 1e-10)

    # Weighted sum over all nodes (Eq. 20)
    scores = ld * L_d + ln * L_n + lx * L_x
    return scores.sum(), scores.detach(), L_d.detach(), L_n.detach(), L_x.detach()

print("Loss function defined.")


Loss function defined.


---
## 10. Training Function (Section 5.1.2)

- 200 epochs, lr=0.005, Adam, 2-layer encoder with hidden_dim=32
- Evaluation: AUC-ROC
- 10 runs with different seeds, report mean +/- STD

In [1052]:
def train_grasped(adj, features, labels, K, beta, S, ld, ln, lx,
                  hidden_dim=32, epochs=200, lr=0.005, seed=42, verbose=True):
    """
    Train GRASPED following Section 5.1.2 exactly.
    Returns: (final_auc, per_node_scores)
    """
    torch.manual_seed(seed)
    np.random.seed(seed)
    n, d = features.shape
    t0 = time.time()
    # # Normalise features (zero mean, unit std) for numerical stability REMOVED
    feat_mean = features.mean(axis=0, keepdims=True)
    feat_std  = features.std(axis=0, keepdims=True) + 1e-8
    features  = (features - feat_mean) / feat_std
    
    
    # Step 1: Eigendecomposition
    if verbose: print(f"  Eigendecomposition ({n}x{n})...")
    L_sp, evals, evecs = compute_laplacian_and_eigen(adj)
    L_t = sparse_to_torch(L_sp).to(device)
    if verbose: print(f"  Done in {time.time()-t0:.1f}s")
    
    # Step 2: Ground truth targets
    deg_np = np.array(adj.sum(axis=1)).flatten().astype(np.float32)
    nmu_np, nls_np = compute_neighbor_stats(adj, features, S=S)
    
    X = torch.tensor(features, dtype=torch.float32).to(device)
    deg_t = torch.tensor(deg_np).to(device)
    nmu_t = torch.tensor(nmu_np).to(device)
    nls_t = torch.tensor(nls_np).to(device)
    
    # Step 3: Build model
    model = GRASPED(d, hidden_dim, K, beta, evals, evecs).to(device)
    # Vanilla Adam optimizer as specified in Section 5.1.2
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    
    # Step 4: Train
    for ep in range(epochs):
        model.train()
        opt.zero_grad()
        loss, _, _, _, _ = compute_losses(model(X, L_t), X, deg_t, nmu_t, nls_t, ld, ln, lx)
        loss.backward()
        #torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        opt.step()
        
        if verbose and (ep + 1) % 50 == 0:
            model.eval()
            with torch.no_grad():
                _, sc, d, n, x = compute_losses(model(X, L_t), X, deg_t, nmu_t, nls_t, ld, ln, lx)
                # Inside your training loop, before line 43
                if torch.isnan(sc).any():
                    print(f"!!! NaN detected in scores at epoch {ep}")
                    # Print model weights or gradients to see if they exploded
                    for name, param in model.named_parameters():
                        if torch.isnan(param).any():
                            print(f"NaN found in layer: {name}")
                    break # Stop the run so you can inspect variables
                auc = roc_auc_score(labels, sc.cpu().numpy())
                print(f"    Epoch {ep+1:3d}/{epochs} | Loss: {loss.item():.4f} | AUC: {auc:.4f} | Degree loss: {d.mean().item():.4f} | Neighbor loss: {n.mean().item():.4f} | Attribute loss: {x.mean().item():.4f}")
    
    # Step 5: Final eval
    model.eval()
    with torch.no_grad():
        _, sc, _, _, _ = compute_losses(model(X, L_t), X, deg_t, nmu_t, nls_t, ld, ln, lx)
        final_auc = roc_auc_score(labels, sc.cpu().numpy())
    if verbose: print(f"    Final AUC: {final_auc:.4f} ({time.time()-t0:.1f}s), ld={ld}, ln={ln}, lx={lx}")
    return final_auc, sc.cpu().numpy()

print("Training function defined.")

Training function defined.


---
## 11. Hyperparameters (Table 4) & Expected Results (Table 3)

In [1053]:
HYPERPARAMS = {
    'disney': {'K': 8,   'beta': 1.2, 'S': 20, 'ln': 0.6, 'lx': 3.0, 'ld': 0.0},
    'books':  {'K': 128, 'beta': 1.5, 'S': 20, 'ln': 3.0, 'lx': 6.0, 'ld': 0.05},
    'reddit': {'K': 16,  'beta': 0.5, 'S': 35, 'ln': 0.2, 'lx': 6.0, 'ld': 0.0},
    'weibo':  {'K': 8,   'beta': 0.5, 'S': 45, 'ln': 0.0, 'lx': 4.0, 'ld': 0.0},
    'enron':  {'K': 16,  'beta': 0.5, 'S': 20, 'ln': 0.4, 'lx': 3.0, 'ld': 0.0},
}

EXPECTED = {
    'weibo':  (82.1, 1.3), 'reddit': (59.9, 0.2), 'disney': (83.0, 3.6),
    'books':  (69.1, 2.6), 'enron':  (83.3, 3.9),
}

print("Hyperparameters configured from Table 4.")

Hyperparameters configured from Table 4.


---
## 12. Run Experiments

In [1054]:
def run_experiment(dataset_name, n_runs=10):
    """Full experiment: n_runs with different seeds, report mean +/- STD."""
    print(f"\n{'='*60}")
    print(f" {dataset_name.upper()}")
    print(f"{'='*60}")
    
    adj, features, labels = load_dataset_pygod(dataset_name)
    #print(f"  NaN values in labels: {np.isnan(labels).sum()}")  # print if NaN values exist in labels
    p = HYPERPARAMS[dataset_name]
    print(f"  Params: K={p['K']}, beta={p['beta']}, S={p['S']}, ld={p['ld']}, ln={p['ln']}, lx={p['lx']}")
    
    aucs = []
    for run in range(n_runs):
        seed = run  # seeds 0,1,...,9 matching paper's "10 different seeds"
        v = True
        if v: print(f"\n  Run {run+1}/{n_runs} (detailed):")
        auc, _ = train_grasped(adj, features, labels, p['K'], p['beta'], p['S'],
                               p['ld'], p['ln'], p['lx'], seed=seed, verbose=v)
        aucs.append(auc * 100)
        if not v: print(f"  Run {run+1}/{n_runs}: AUC = {auc*100:.1f}%")
    
    m, s = np.mean(aucs), np.std(aucs)
    em, es = EXPECTED[dataset_name]
    print(f"\n  Result:   {m:.1f} +/- {s:.1f}")
    print(f"  Expected: {em:.1f} +/- {es:.1f}")
    return m, s

In [1055]:
# Quick validation on Disney (124 nodes, takes seconds)
run_experiment('disney', n_runs=10)


 DISNEY
  Params: K=8, beta=1.2, S=20, ld=0.0, ln=0.6, lx=3.0

  Run 1/10 (detailed):
  Eigendecomposition (124x124)...
  Done in 0.0s
    Epoch  50/200 | Loss: 22.6011 | AUC: 0.4054 | Degree loss: 0.0720 | Neighbor loss: 0.0083 | Attribute loss: 0.0555
    Epoch 100/200 | Loss: 9.7990 | AUC: 0.3390 | Degree loss: 0.0737 | Neighbor loss: 0.0081 | Attribute loss: 0.0225
    Epoch 150/200 | Loss: 7.9679 | AUC: 0.3517 | Degree loss: 0.0753 | Neighbor loss: 0.0081 | Attribute loss: 0.0197
    Epoch 200/200 | Loss: 7.4659 | AUC: 0.3178 | Degree loss: 0.0774 | Neighbor loss: 0.0081 | Attribute loss: 0.0188
    Final AUC: 0.3178 (4.3s), ld=0.0, ln=0.6, lx=3.0

  Run 2/10 (detailed):
  Eigendecomposition (124x124)...
  Done in 0.0s
    Epoch  50/200 | Loss: 16.0138 | AUC: 0.3602 | Degree loss: 0.0777 | Neighbor loss: 0.0087 | Attribute loss: 0.0403
    Epoch 100/200 | Loss: 8.6268 | AUC: 0.3249 | Degree loss: 0.0809 | Neighbor loss: 0.0081 | Attribute loss: 0.0208
    Epoch 150/200 | Loss: 7.

(np.float64(32.6271186440678), np.float64(3.0757251023716745))

In [1056]:
# Books (1,418 nodes)
#run_experiment('books', n_runs=3)

In [1057]:
# # Full experiments (10 runs each, as in paper)
# # Large datasets need eigendecomp of 8K-13K matrices - takes minutes each
# results = {}
# for name in ['disney']:  # add 'reddit','weibo','enron' when ready
#     results[name] = run_experiment(name, n_runs=10)

In [1058]:
# Uncomment for larger datasets (need more time + memory):
# for name in ['reddit', 'weibo', 'enron']:
#     results[name] = run_experiment(name, n_runs=10)

In [1059]:
# Final comparison
print("\n" + "="*65)
print(" RESULTS: Our Implementation vs Paper (Table 3)")
print("="*65)
print(f"{'Dataset':<10} {'Ours':<18} {'Paper':<18}")
print("-"*65)
for name in ['weibo', 'reddit', 'disney', 'books', 'enron']:
    em, es = EXPECTED[name]
    paper = f"{em:.1f} +/- {es:.1f}"
    ours = f"{results[name][0]:.1f} +/- {results[name][1]:.1f}" if name in results else "(not run)"
    print(f"{name:<10} {ours:<18} {paper:<18}")
print("="*65)
print("\nNote: Exact match requires real PyGOD datasets, not synthetic.")


 RESULTS: Our Implementation vs Paper (Table 3)
Dataset    Ours               Paper             
-----------------------------------------------------------------
weibo      (not run)          82.1 +/- 1.3      
reddit     (not run)          59.9 +/- 0.2      
disney     (not run)          83.0 +/- 3.6      
books      (not run)          69.1 +/- 2.6      
enron      (not run)          83.3 +/- 3.9      

Note: Exact match requires real PyGOD datasets, not synthetic.


In [1060]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score
import torch


def evaluate_once(
        test_results,
        prices,
        forward_window,
        crash_threshold
):

    y_true = []
    y_scores = []

    sorted_dates = sorted(test_results.keys())

    valid_dates = [
        d for d in sorted_dates
        if d <= prices.index[-1] - pd.Timedelta(days=forward_window)
    ]

    for t in valid_dates:

        stocks, signals, *_ = test_results[t]  # ignore feat/struct signals

        if isinstance(signals, torch.Tensor):
            signals = signals.cpu().numpy()

        signals = signals.flatten()

        available = [s for s in stocks if s in prices.columns]

        if len(available) == 0:
            continue

        mask = [i for i,s in enumerate(stocks) if s in available]

        signals = signals[mask]

        p_t = prices.loc[t, available]

        future_idx = prices.index.searchsorted(
            t + pd.Timedelta(days=forward_window)
        )

        if future_idx >= len(prices):
            continue

        future_date = prices.index[future_idx]

        p_future = prices.loc[future_date, available]

        fwd_returns = (p_future - p_t) / p_t

        crash = (fwd_returns < crash_threshold).astype(int)

        # get rid of inf in signals
        signals = np.nan_to_num(signals, nan=0.0, posinf=1e6, neginf=-1e6)

        y_true.extend(crash.values)

        y_scores.extend(signals)

    if len(y_true) == 0:
        return None

    y_true = np.array(y_true)
    y_scores = np.array(y_scores)

    auc = roc_auc_score(y_true, y_scores)

    baseline = y_true.mean()

    precision = y_true[y_scores > np.percentile(y_scores, 90)].mean()

    lift = precision / baseline if baseline > 0 else np.nan

    return auc, lift, baseline

def grid_search(
        test_results,
        prices,
        forward_windows,
        crash_thresholds
):

    rows = []

    for fw in forward_windows:

        for ct in crash_thresholds:

            result = evaluate_once(
                test_results,
                prices,
                fw,
                ct
            )

            if result is None:
                continue

            auc, lift, baseline = result

            rows.append({

                "ForwardWindow": fw,

                "CrashThreshold": ct,

                "AUC": auc,

                "Lift": lift,

                "Baseline": baseline

            })

            print(
                f"FW={fw:3d} "
                f"CT={ct:6.2f} "
                f"AUC={auc:.3f} "
                f"Lift={lift:.2f}"
            )

    return pd.DataFrame(rows)


In [1064]:
import pandas as pd
for alpha in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]:
    print(f"\n{'='*60}\nALPHA = {alpha}\n{'='*60}")
    test_results = pd.read_pickle(f"test_results_test_1Apr_alpha_{alpha}.pkl")
    test_prices = pd.read_pickle(f"test_prices_test_1Apr_alpha_{alpha}.pkl")

    forward_windows = [5,10,22,44,66,120]

    crash_thresholds = [-0.05,-0.10,-0.15,-0.20,-0.30]


    df_results = grid_search(

        test_results,

        test_prices,

        forward_windows,

        crash_thresholds

    )
# for test_results_test : FW=  5 CT= -0.30 AUC=0.785 Lift=4.45 testing from 2020-2024



ALPHA = 0.0
FW=  5 CT= -0.05 AUC=0.576 Lift=1.54
FW=  5 CT= -0.10 AUC=0.633 Lift=2.42
FW=  5 CT= -0.15 AUC=0.692 Lift=3.47
FW=  5 CT= -0.20 AUC=0.730 Lift=4.11
FW=  5 CT= -0.30 AUC=0.699 Lift=3.42
FW= 10 CT= -0.05 AUC=0.550 Lift=1.24
FW= 10 CT= -0.10 AUC=0.589 Lift=1.70
FW= 10 CT= -0.15 AUC=0.646 Lift=2.47
FW= 10 CT= -0.20 AUC=0.695 Lift=3.21
FW= 10 CT= -0.30 AUC=0.701 Lift=3.31
FW= 22 CT= -0.05 AUC=0.527 Lift=1.03
FW= 22 CT= -0.10 AUC=0.547 Lift=1.14
FW= 22 CT= -0.15 AUC=0.560 Lift=1.29
FW= 22 CT= -0.20 AUC=0.570 Lift=1.48
FW= 22 CT= -0.30 AUC=0.597 Lift=1.86
FW= 44 CT= -0.05 AUC=0.502 Lift=0.87
FW= 44 CT= -0.10 AUC=0.514 Lift=0.89
FW= 44 CT= -0.15 AUC=0.520 Lift=0.92
FW= 44 CT= -0.20 AUC=0.514 Lift=0.92
FW= 44 CT= -0.30 AUC=0.518 Lift=1.07
FW= 66 CT= -0.05 AUC=0.490 Lift=0.81
FW= 66 CT= -0.10 AUC=0.493 Lift=0.78
FW= 66 CT= -0.15 AUC=0.494 Lift=0.76
FW= 66 CT= -0.20 AUC=0.494 Lift=0.76
FW= 66 CT= -0.30 AUC=0.485 Lift=0.74
FW=120 CT= -0.05 AUC=0.492 Lift=0.81
FW=120 CT= -0.10 AUC=0.49